# Monte Carlo report

Compares **conditions across seeds** for an experiment written by `beaversim.experiments.run_montecarlo`
(`<experiment>/<condition>/seed_<n>/` plus `experiment.json`). Lines are the mean across seeds, patterned bands
±1σ; each line is labelled at its end with the condition and the number of seeds.

In [ ]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "beaversim").is_dir())
sys.path.insert(0, str(root))

from beaversim import viz
from beaversim.records import load_experiment

## Parameters

In [ ]:
EXPERIMENT_DIR = root / "output/simulations/<experiment>"
LABELS = {}             # optional display names, e.g. {"visits": "stigmergic"}
SUBTITLE = None         # e.g. "10 agents · Rumney Ranch A2"
MEDIUM = "chapter"      # "chapter", "paper" or "slides"
PALETTE = None          # None = default, or a path to a palette JSON from the designers
SAVE = False            # write each figure as PDF + PNG into FIG_DIR
FIG_DIR = root / "output/figures" / EXPERIMENT_DIR.name

In [ ]:
viz.use_theme(MEDIUM, PALETTE)
exp = load_experiment(EXPERIMENT_DIR)


def show(fig, name):
    """Display ``fig`` and optionally save it under FIG_DIR."""
    if SAVE:
        viz.save(fig, FIG_DIR / name)
    return fig

## What was compared
One row per condition: seeds, duration, colony size and the config overrides that define the condition.

In [ ]:
import pandas as pd
pd.DataFrame(viz.montecarlo.summary(exp)).set_index("condition")

## 1. Spatial diffusion
$\mathcal{D}$: number of cells ever visited, per role.

In [ ]:
show(viz.montecarlo.over_time(exp, "diffusion", labels=LABELS, subtitle=SUBTITLE), "diffusion");

## 2. Canalization ratio
$\mathcal{C}$: share of active traffic on the top 5% busiest cells, per role. Smoothed with a 1-week moving
average; the first day is hidden because $\mathcal{C}$ jumps between 0 and 1 while almost no cells carry traffic.

In [ ]:
show(viz.montecarlo.over_time(exp, "canalization", labels=LABELS, subtitle=SUBTITLE), "canalization");

## 3. Environment
Share of open-water cells and mean vegetation quality.

In [ ]:
show(viz.montecarlo.over_time(exp, "water", labels=LABELS, subtitle=SUBTITLE), "water");

In [ ]:
show(viz.montecarlo.over_time(exp, "vegetation", labels=LABELS, subtitle=SUBTITLE), "vegetation");

## 4. Representative runs
Cumulative trail traffic of the first seed of each condition. Open `run_report.ipynb` on any run directory
for the full single-run view.

In [ ]:
for cond, runs in exp:
    show(viz.single.trail_traffic(runs[0], title=f"trail traffic · {LABELS.get(cond, cond)}"), f"traffic_{cond}")